NAME : Yash Bisen |    PRN - 202301040160

LAB TITLE - LSTM Next Word Prediction

GROUP - Varad Kolhe , Tanmay Falke , Ishwari Mahajan

Github - https://github.com/yash240804/deep_learning_yolo

Date of submission - 11/04/2026


# 🧠 End-to-End LSTM Next Word Prediction
Welcome to this comprehensive guide on building a Next Word Prediction model using a Long Short-Term Memory (LSTM) neural network.

### 📐 The Mathematics of LSTM
Standard Recurrent Neural Networks (RNNs) suffer from the vanishing gradient problem, making it hard to learn long-term dependencies. LSTMs solve this using a cell state regulated by three gates. At any timestep $t$:

1. **Forget Gate ($f_t$):** Decides what to discard from the previous state.
$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$$
2. **Input Gate ($i_t$) & Candidate State ($\tilde{C}_t$):** Decides what new information to add.
$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$$
$$\tilde{C}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)$$
3. **Cell State Update ($C_t$):** The new memory of the cell.
$$C_t = f_t * C_{t-1} + i_t * \tilde{C}_t$$
4. **Output Gate ($o_t$) & Hidden State ($h_t$):** Decides what to output based on the cell state.
$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$$
$$h_t = o_t * \tanh(C_t)$$

Let's start by installing and importing the required libraries.

In [ ]:
# Install datasets if not already available in the Colab environment
!pip install datasets -q

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from datasets import load_dataset
import pickle
import re

print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.19.0


## 📊 1. Data Collection
We are using the **Wikitext-2** dataset from Hugging Face. It contains high-quality, verified articles from Wikipedia, making it excellent for language modeling.

To ensure the training runs smoothly within Colab's standard RAM limits (especially during the one-hot encoding phase), we will subset the data to the first 1,500 lines.

In [ ]:
print("Downloading Wikitext-2 dataset...")
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# Subsetting to prevent Out-Of-Memory (OOM) errors on free Colab tiers
text_data = dataset['text'][:1500]

print(f"\nLoaded {len(text_data)} lines of text.")
print("Sample raw text:")
print(text_data[15:18])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]


Loaded 1500 lines of text.
Sample raw text:
[' The game takes place during the Second Europan War . Gallian Army Squad 422 , also known as " The Nameless " , are a penal military unit composed of criminals , foreign deserters , and military offenders whose real names are erased from the records and thereon officially referred to by numbers . Ordered by the Gallian military to perform the most dangerous missions that the Regular Army and Militia will not do , they are nevertheless up to the task , exemplified by their motto , Altaha Abilia , meaning " Always Ready . " The three main characters are No.7 Kurt Irving , an army officer falsely accused of treason who wishes to redeem himself ; Ace No.1 Imca , a female Darcsen heavy weapons specialist who seeks revenge against the Valkyria who destroyed her home ; and No.13 Riela Marcellis , a seemingly jinxed young woman who is unknowingly a descendant of the Valkyria . Together with their fellow squad members , these three are tasked to fi

## 🧹 2. Data Preprocessing
Raw text contains capitalization, punctuation, and irregular spacing that confuse the model.

We will create a function to:
* Convert all text to lowercase.
* Remove non-alphabetic characters (punctuation, numbers).
* Strip extra whitespaces.
* Filter out empty lines or lines that are too short to form meaningful sequences.

In [ ]:
def clean_text(texts):
    cleaned = []
    for text in texts:
        text = text.lower()
        # Keep only lowercase letters and spaces
        text = re.sub(r'[^a-z\s]', '', text)
        # Collapse multiple spaces into one
        text = re.sub(r'\s+', ' ', text).strip()
        # Only keep sentences with more than 3 words (approx > 15 chars)
        if len(text) > 15:
            cleaned.append(text)
    return cleaned

cleaned_corpus = clean_text(text_data)

print(f"Corpus size after cleaning: {len(cleaned_corpus)} sentences.")
print("\nSample cleaned text:")
print(cleaned_corpus[:3])

Corpus size after cleaning: 800 sentences.

Sample cleaned text:
['valkyria chronicles iii', 'senj no valkyria unrecorded chronicles japanese lit valkyria of the battlefield commonly referred to as valkyria chronicles iii outside japan is a tactical role playing video game developed by sega and mediavision for the playstation portable released in january in japan it is the third game in the valkyria series employing the same fusion of tactical and real time gameplay as its predecessors the story runs parallel to the first game and follows the nameless a penal military unit serving the nation of gallia during the second europan war who perform secret black operations and are pitted against the imperial unit calamaty raven', 'the game began development in carrying over a large portion of the work done on valkyria chronicles ii while it retained the standard features of the series it also underwent multiple adjustments such as making the game more forgiving for series newcomers character 

## 🔠 3. Tokenization and Sequence Generation
Neural networks cannot process raw text; they require numbers.
1. **Tokenization:** We use Keras `Tokenizer` to assign a unique integer to every word in our vocabulary.
2. **N-gram Sequences:** We break our sentences into expanding sequences.
   * Example: `"i love machine learning"` becomes `["i love", "i love machine", "i love machine learning"]`.
3. **Padding:** Because sequences have different lengths, we pad them with zeros at the beginning (`pre` padding) so they are all the same length.

In [ ]:
# Initialize and fit tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(cleaned_corpus)
total_words = len(tokenizer.word_index) + 1 # +1 for the padding token '0'

print(f"Total Vocabulary Size: {total_words}")

# Generate n-gram sequences
input_sequences = []
for line in cleaned_corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print(f"Total N-gram sequences generated: {len(input_sequences)}")

# Pad sequences
max_sequence_len = max([len(seq) for seq in input_sequences])
print(f"Maximum sequence length: {max_sequence_len}")

input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

# Split into Features (X) and Labels (y)
input_sequences = np.array(input_sequences)
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# One-hot encode the labels
y = to_categorical(y, num_classes=total_words)

print(f"Shape of Features (X): {X.shape}")
print(f"Shape of Labels (y): {y.shape}")

Total Vocabulary Size: 9930
Total N-gram sequences generated: 69269
Maximum sequence length: 462
Shape of Features (X): (69269, 461)
Shape of Labels (y): (69269, 9930)


## 🏗️ 4. Model Architecture & Training
We will construct a Sequential model:
1. **Embedding Layer:** Converts our integer word tokens into dense vectors of fixed size (100 dimensions). It learns semantic relationships between words.
2. **LSTM Layer:** Contains 150 units. It processes the sequence of word embeddings, carrying context forward using its internal state.
3. **Dense Layer:** Uses a `softmax` activation to output a probability distribution across the entire vocabulary. The word with the highest probability is our prediction.

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=total_words, output_dim=100, input_length=max_sequence_len-1))
model.add(LSTM(150, return_sequences=False))
# Adding dropout to prevent overfitting on this small dataset
model.add(Dropout(0.2))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

print("\n--- Commencing Training ---")
# Training for 40 epochs. In Colab with a GPU, this should take a few minutes.
history = model.fit(X, y, epochs=40, batch_size=128, verbose=1)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


--- Commencing Training ---
Epoch 1/40
542/542 ━━━━━━━━━━━━━━━━━━━━ 32s 41ms/step - accuracy: 0.0810 - loss: 7.2583
Epoch 2/40
542/542 ━━━━━━━━━━━━━━━━━━━━ 23s 42ms/step - accuracy: 0.0997 - loss: 6.7896
Epoch 3/40
542/542 ━━━━━━━━━━━━━━━━━━━━ 24s 43ms/step - accuracy: 0.1148 - loss: 6.5421
Epoch 4/40
542/542 ━━━━━━━━━━━━━━━━━━━━ 24s 44ms/step - accuracy: 0.1273 - loss: 6.3196
Epoch 5/40
542/542 ━━━━━━━━━━━━━━━━━━━━ 25s 45ms/step - accuracy: 0.1387 - loss: 6.0944
Epoch 6/40
542/542 ━━━━━━━━━━━━━━━━━━━━ 25s 46ms/step - accuracy: 0.1492 - loss: 5.8731
Epoch 7/40
542/542 ━━━━━━━━━━━━━━━━━━━━ 24s 45ms/step - accuracy: 0.1599 - loss: 5.6509
Epoch 8/40
362/542 ━━━━━━━━━━━━━━━━━━━━ 8s 47ms/step - accuracy: 0.1761 - loss: 5.3574

KeyboardInterrupt: 

## 💾 5. Prediction Function & Artifact Saving
Now that the model is trained, we define a helper function to format new text strings identically to our training data, feed them into the model, and decode the predicted integer back into a word.

We also save the model `.h5` file and the `tokenizer.pickle` file. These are the exact files you will load into your FastAPI backend later.

In [ ]:
def predict_next_word(seed_text):
    # Preprocess text exactly like training data
    cleaned_text = seed_text.lower()
    cleaned_text = re.sub(r'[^a-z\s]', '', cleaned_text)

    # Convert to sequence and pad
    token_list = tokenizer.texts_to_sequences([cleaned_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

    # Get probabilities and find the highest one
    predicted_probs = model.predict(token_list, verbose=0)
    predicted_index = np.argmax(predicted_probs, axis=-1)[0]

    # Reverse map index to word
    for word, index in tokenizer.word_index.items():
        if index == predicted_index:
            return word
    return "<UNKNOWN>"

# Save Model and Tokenizer for production
model.save("lstm_next_word_model.h5")
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open('max_seq_len.txt', 'w') as f:
    f.write(str(max_sequence_len))

print("Artifacts saved to Colab environment!")

## 🧪 6. Model Evaluation (15 Test Cases)
Below are 15 individual test cells. Run them sequentially to test how the model responds to different contextual prompts based on the Wikipedia text it learned from.

In [ ]:
seed_1 = "the history of"
print(f"Input: '{seed_1}'")
print(f"Prediction: {predict_next_word(seed_1)}")

In [ ]:
seed_2 = "he was born in"
print(f"Input: '{seed_2}'")
print(f"Prediction: {predict_next_word(seed_2)}")

In [ ]:
seed_3 = "during the war they"
print(f"Input: '{seed_3}'")
print(f"Prediction: {predict_next_word(seed_3)}")

In [ ]:
seed_4 = "she decided to"
print(f"Input: '{seed_4}'")
print(f"Prediction: {predict_next_word(seed_4)}")

In [ ]:
seed_5 = "the capital city is"
print(f"Input: '{seed_5}'")
print(f"Prediction: {predict_next_word(seed_5)}")

In [ ]:
seed_6 = "in the early years"
print(f"Input: '{seed_6}'")
print(f"Prediction: {predict_next_word(seed_6)}")

In [ ]:
seed_7 = "the theory of"
print(f"Input: '{seed_7}'")
print(f"Prediction: {predict_next_word(seed_7)}")

In [ ]:
seed_8 = "it is widely believed"
print(f"Input: '{seed_8}'")
print(f"Prediction: {predict_next_word(seed_8)}")

In [ ]:
seed_9 = "a large number of"
print(f"Input: '{seed_9}'")
print(f"Prediction: {predict_next_word(seed_9)}")

In [ ]:
seed_10 = "the most famous"
print(f"Input: '{seed_10}'")
print(f"Prediction: {predict_next_word(seed_10)}")

In [ ]:
seed_11 = "after a long period"
print(f"Input: '{seed_11}'")
print(f"Prediction: {predict_next_word(seed_11)}")

In [ ]:
seed_12 = "he became the first"
print(f"Input: '{seed_12}'")
print(f"Prediction: {predict_next_word(seed_12)}")

In [ ]:
seed_13 = "many people think that"
print(f"Input: '{seed_13}'")
print(f"Prediction: {predict_next_word(seed_13)}")

In [ ]:
seed_14 = "the primary reason for"
print(f"Input: '{seed_14}'")
print(f"Prediction: {predict_next_word(seed_14)}")

In [ ]:
seed_15 = "at the end of"
print(f"Input: '{seed_15}'")
print(f"Prediction: {predict_next_word(seed_15)}")

## 🤖 Acknowledgements

This notebook and its associated implementation were developed with the assistance of **Gemini**.

Specifically, the AI was utilized to accelerate and improve the following areas of the project:
* **Text Box Content Refinement:** Structuring and enhancing the clarity of the explanatory markdown sections and mathematical breakdowns.
* **Test Case Generation:** Formulating the 15 diverse and contextually relevant seed phrases to rigorously evaluate the model's predictive capabilities.
* **Code Fixes & Optimization:** Ensuring the Python implementation, Keras model architecture, and sequence processing pipelines adhere to industry best practices and remain bug-free.